In [11]:
import random
from pokerkit import Automation, NoLimitTexasHoldem

# Create the state for 2 players (e.g., Human vs. Agent)
state = NoLimitTexasHoldem.create_state(
    (
        Automation.ANTE_POSTING,
        Automation.BET_COLLECTION,
        Automation.BLIND_OR_STRADDLE_POSTING,
        Automation.CARD_BURNING,
        Automation.HOLE_CARDS_SHOWING_OR_MUCKING,
        Automation.HAND_KILLING,
        Automation.CHIPS_PUSHING,
        Automation.CHIPS_PULLING,
    ),
    True,      # Uniform antes
    0,       # Ante amount
    (1000, 2000), # Small/Big blind
    2000,      # Min-bet
    (100000, 100000), # Starting stacks (Human, Agent)
    2,         # Number of players
)

In [20]:

# 2. Start the hand by dealing hole cards safely
while state.can_deal_hole():
    state.deal_hole()
# 2. The Decision Loop (Per-Turn) [cite: 47, 50]
while state.status:
    if state.can_deal_board():
        state.deal_board()
        print(f"\n--- NEW STREET: {state.street_index} ---")
        print(f"Board: {state.board_cards}")
    # Check if a player needs to act
    if state.actor_index is not None:
        current_player = state.actor_index
        
        if current_player == 0:  # HUMAN PLAYER
            print("\n===============================")
            print(f"BOARD: {state.board_cards if state.board_cards else 'PRE-FLOP'}")
            print(f"YOUR HAND: {state.hole_cards[0]}")
            print(f"POT: {state.total_pot_amount}")
            print("===============================")
            # Capture human input to update the game state
            action = input("Action (f/c/r): ")
            if action == 'f': state.fold()
            elif action == 'r':
                raise_amount = int(input("Enter total amount to raise to: "))
                state.complete_bet_or_raise_to(raise_amount)
            else: state.check_or_call()
            
        else:  # AGENT PLAYER (LLM Core + Tools) [cite: 19]
            # STEP 1: Tool Use - Calculate Equity [cite: 22, 39, 48]
            # E = Pot * (P(win) + 1/n * P(draw)) [cite: 24]
            
            # STEP 2: Memory Access - Read Short-term Context [cite: 44, 48]
            # STEP 3: LLM Reasoning - Form Range Hypothesis [cite: 41, 48]
            
            print("Agent is thinking...")
            state.check_or_call() # Placeholder for LLM decision
            
    # 3. Environment Progression (Dealing Streets)
    if state.can_deal_board():
        state.deal_board()

# 4. Post-Hand Reflection (Per-Hand) [cite: 51, 52]
print(f"Hand Over. Payoffs: {state.payoffs}")
# Agent observes outcome and updates Table Memory/Self-Profile [cite: 52, 54]

Hand Over. Payoffs: [7000, -7000]


In [22]:
# 1. Setup the Environment
state = NoLimitTexasHoldem.create_state(
    # REMOVE almost all automations to gain control
    (
        Automation.ANTE_POSTING,
        Automation.BLIND_OR_STRADDLE_POSTING,
    ), 
    True, 0, (1000, 2000), 2000, (100000, 100000), 2
)

# 2. The Training/Simulation Loop
while state.status:
    # DEALING PHASE: If the engine needs cards, give them
    if state.can_deal_hole():
        state.deal_hole()
    elif state.can_deal_board():
        state.deal_board()
    
    # ACTION PHASE: If a player (Agent or Opponent) needs to move
    elif state.actor_index is not None:
        obs = {
            "pot": state.total_pot_amount,
            "board": state.board_cards,
            "hole": state.hole_cards[state.actor_index],
            "history": state.operations # This is your 'Short-term Memory' [cite: 31]
        }
        
        # This is where you feed 'obs' to your LLM Core [cite: 36]
        # For now, we use a simple Tool-only Bot logic [cite: 99]
        if state.actor_index == 1: # Agent
            # STEP 1: Run Equity Tool [cite: 22, 39]
            # STEP 2: LLM Core decides [cite: 19]
            state.check_or_call() 
        else: # Opponent (The 'Human' or 'Random' player)
            state.check_or_call()
            
    else:
        # If no one can act and no cards can be dealt, the hand is over
        break

In [23]:
obs

{'pot': 4000,
 'board': [],
 'hole': [2s, 3h],
 'history': [BlindOrStraddlePosting(commentary=None, player_index=0, amount=2000),
  BlindOrStraddlePosting(commentary=None, player_index=1, amount=1000),
  HoleDealing(commentary=None, player_index=0, cards=(2s,), statuses=(False,)),
  HoleDealing(commentary=None, player_index=1, cards=(Th,), statuses=(False,)),
  HoleDealing(commentary=None, player_index=0, cards=(3h,), statuses=(False,)),
  HoleDealing(commentary=None, player_index=1, cards=(9s,), statuses=(False,)),
  CheckingOrCalling(commentary=None, player_index=1, amount=1000),
  CheckingOrCalling(commentary=None, player_index=0, amount=0)]}

In [26]:
while state.status:
    # 1. Check if the engine is waiting for cards (Flop/Turn/River)
    if state.can_deal_board():
        state.deal_board()
        print(f"\n--- DEALING {state.street_index} ---")
        continue # Restart loop to check for new player actions on the new street

    # 2. If it's a player's turn, get their observation and action
    if state.actor_index is not None:
        obs = {
            "pot": state.total_pot,
            "board": state.board_cards,
            "hole": state.hole_cards[state.actor_index],
            "history": state.operations 
        }
        
        # DECISION LOGIC
        if state.actor_index == 1:
            state.check_or_call() 
        else:
            # For now, just call/check to see the board move
            state.check_or_call()
            
    else:
        # If no one can act and no cards can be dealt, we are truly finished
        break

obs

{'pot': 4000,
 'board': [],
 'hole': [2s, 3h],
 'history': [BlindOrStraddlePosting(commentary=None, player_index=0, amount=2000),
  BlindOrStraddlePosting(commentary=None, player_index=1, amount=1000),
  HoleDealing(commentary=None, player_index=0, cards=(2s,), statuses=(False,)),
  HoleDealing(commentary=None, player_index=1, cards=(Th,), statuses=(False,)),
  HoleDealing(commentary=None, player_index=0, cards=(3h,), statuses=(False,)),
  HoleDealing(commentary=None, player_index=1, cards=(9s,), statuses=(False,)),
  CheckingOrCalling(commentary=None, player_index=1, amount=1000),
  CheckingOrCalling(commentary=None, player_index=0, amount=0)]}

In [ ]:
def tool_only_policy(observation):
    # 1. Use Monte Carlo to get P(win) based on 'hole' and 'board' [cite: 25]
    win_prob = calculate_monte_carlo_equity(observation['hole'], observation['board'])
    
    # 2. Basic mathematical decision [cite: 92]
    if win_prob > 0.6:
        return "raise"
    elif win_prob > 0.3:
        return "call"
    else:
        return "fold"